In [1]:
import re
import string
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from snowballstemmer import stemmer

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import FunctionTransformer


In [2]:

TRAIN_PATH = Path("../data/processed/train.csv")

if not TRAIN_PATH.exists():
    raise FileNotFoundError(f"Training CSV not found at {TRAIN_PATH.resolve()}. Run the notebook from its project notebooks directory or update TRAIN_PATH.")

train_df = pd.read_csv(TRAIN_PATH)

X_train = train_df["review"]
y_train = train_df["sentiment"]

print("Training reviews:", len(X_train))
print("Training labels:", y_train.value_counts())

Training reviews: 40006
Training labels: sentiment
positive    20004
negative    20002
Name: count, dtype: int64


In [3]:
def strip_html_batch(texts):
    return [
        BeautifulSoup(
            "" if pd.isna(text) else str(text),
            "html.parser"
        ).get_text(separator=" ")
        for text in texts
    ]


baseline = Pipeline([
    (
        "html",
        FunctionTransformer(
            strip_html_batch,
            validate=False
        )
    ),
    (
        "vectorizer",
        CountVectorizer()
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

print("Baseline pipeline created successfully!")

Baseline pipeline created successfully!


In [4]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    baseline,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

print("Fold accuracies:", np.round(scores, 4))
print(f"Mean CV accuracy: {scores.mean():.4f}")
print(f"Standard deviation: {scores.std():.4f}")

Fold accuracies: [0.8865 0.8859 0.8913 0.8855 0.8895]
Mean CV accuracy: 0.8877
Standard deviation: 0.0023


In [5]:
def lowercase_batch(texts):
    return [text.lower() for text in texts]


def remove_punctuation_batch(texts):
    return [
        re.sub(r"[^\w\s]", " ", text)
        for text in texts
    ]


# Keep negation words to preserve sentiment
custom_stop_words = list(
    ENGLISH_STOP_WORDS - {"not", "no", "never"}
)


def remove_stopwords_batch(texts):
    cleaned = []

    for text in texts:
        words = text.split()
        words = [
            word for word in words
            if word.lower() not in custom_stop_words
        ]
        cleaned.append(" ".join(words))

    return cleaned


print("Preprocessing functions ready!")

Preprocessing functions ready!


In [6]:

english_stemmer = stemmer("english")

def stem_text_batch(texts):
    return [
        " ".join(english_stemmer.stemWords(str(text).lower().split()))
        for text in texts
    ]

print(stem_text_batch([
    "The movies were amazing and entertaining",
    "The performances were disappointing"
]))

['the movi were amaz and entertain', 'the perform were disappoint']


In [12]:
# Compare preprocessing variants using the same classifier and cross-validation
preprocessing_variants = {
    "Raw text": [],
    "HTML stripping": [
        ("html", FunctionTransformer(strip_html_batch, validate=False))
    ],
    "Lowercasing": [
        ("html", FunctionTransformer(strip_html_batch, validate=False)),
        ("lowercase", FunctionTransformer(lowercase_batch, validate=False))
    ],
    "Punctuation removal": [
        ("html", FunctionTransformer(strip_html_batch, validate=False)),
        ("lowercase", FunctionTransformer(lowercase_batch, validate=False)),
        ("punctuation", FunctionTransformer(remove_punctuation_batch, validate=False))
    ],
    "Stop-word removal": [
        ("html", FunctionTransformer(strip_html_batch, validate=False)),
        ("lowercase", FunctionTransformer(lowercase_batch, validate=False)),
        ("punctuation", FunctionTransformer(remove_punctuation_batch, validate=False)),
        ("stopwords", FunctionTransformer(remove_stopwords_batch, validate=False))
    ],
    "Stemming": [
        ("html", FunctionTransformer(strip_html_batch, validate=False)),
        ("lowercase", FunctionTransformer(lowercase_batch, validate=False)),
        ("punctuation", FunctionTransformer(remove_punctuation_batch, validate=False)),
        ("stemming", FunctionTransformer(stem_text_batch, validate=False))
    ]
}

results = []

for name, preprocessing_steps in preprocessing_variants.items():
    model = Pipeline(
        preprocessing_steps + [
            ("vectorizer", CountVectorizer()),
            ("classifier", LogisticRegression(max_iter=1000, random_state=42))
        ]
    )

    variant_scores = cross_val_score(
        model, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1
    )

    results.append({
        "Variant": name,
        "Mean CV Accuracy": variant_scores.mean(),
        "Std Dev": variant_scores.std()
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

for row in results:
    print(f"{row['Variant']}: {row['Mean CV Accuracy']:.4f} +/- {row['Std Dev']:.4f}")


            Variant  Mean CV Accuracy  Std Dev
           Raw text          0.887742 0.002300
     HTML stripping          0.887742 0.002254
        Lowercasing          0.887742 0.002254
Punctuation removal          0.887742 0.002254
  Stop-word removal          0.883468 0.002731
           Stemming          0.883792 0.001685
Raw text: 0.8877 +/- 0.0023
HTML stripping: 0.8877 +/- 0.0023
Lowercasing: 0.8877 +/- 0.0023
Punctuation removal: 0.8877 +/- 0.0023
Stop-word removal: 0.8835 +/- 0.0027
Stemming: 0.8838 +/- 0.0017


In [8]:

english_stemmer = stemmer("english")
print(english_stemmer.stemWords(["movies", "amazing", "entertaining"]))

['movi', 'amaz', 'entertain']


In [9]:
# Compare raw counts against TF-IDF


vectorizers = {
    "CountVectorizer": CountVectorizer(),
    "TF-IDF": TfidfVectorizer()
}

tfidf_results = {}

for name, vectorizer in vectorizers.items():

    model = Pipeline([
        (
            "html",
            FunctionTransformer(
                strip_html_batch,
                validate=False
            )
        ),
        ("vectorizer", vectorizer),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ])

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )

    tfidf_results[name] = scores

    print(f"\n{name}")
    print("Fold accuracies:", np.round(scores, 4))
    print(f"Mean accuracy: {scores.mean():.4f}")
    print(f"Standard deviation: {scores.std():.4f}")


CountVectorizer
Fold accuracies: [0.8865 0.8859 0.8913 0.8855 0.8895]
Mean accuracy: 0.8877
Standard deviation: 0.0023

TF-IDF
Fold accuracies: [0.8924 0.8945 0.8951 0.8939 0.8951]
Mean accuracy: 0.8942
Standard deviation: 0.0010


In [10]:
# P05: TF-IDF vs. Bag-of-Words Counts


vectorizers = {
    "Raw Counts": CountVectorizer(),
    "TF-IDF": TfidfVectorizer()
}

comparison_results = []

for name, vectorizer in vectorizers.items():

    model = Pipeline([
        (
            "html",
            FunctionTransformer(
                strip_html_batch,
                validate=False
            )
        ),
        ("vectorizer", vectorizer),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ])

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )

    comparison_results.append({
        "Representation": name,
        "Mean CV Accuracy": scores.mean(),
        "Standard Deviation": scores.std()
    })

    print(
        f"{name}: {scores.mean():.4f} "
        f"+/- {scores.std():.4f}"
    )

comparison_df = pd.DataFrame(comparison_results)
comparison_df

Raw Counts: 0.8877 +/- 0.0023
TF-IDF: 0.8942 +/- 0.0010


,Representation,Mean CV Accuracy,Standard Deviation
0,Raw Counts,0.887742,0.002254
1,TF-IDF,0.894216,0.001019


In [11]:
# P05: TF-IDF Unigrams vs Unigrams + Bigrams


ngram_results = []

for name, ngram_range in [
    ("Unigrams", (1, 1)),
    ("Unigrams + Bigrams", (1, 2))
]:

    model = Pipeline([
        (
            "html",
            FunctionTransformer(
                strip_html_batch,
                validate=False
            )
        ),
        (
            "vectorizer",
            TfidfVectorizer(
                ngram_range=ngram_range,
                min_df=2,
                max_features=100000
            )
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ])

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )

    ngram_results.append({
        "Representation": name,
        "Mean CV Accuracy": scores.mean(),
        "Standard Deviation": scores.std()
    })

    print(
        f"{name}: {scores.mean():.4f} "
        f"+/- {scores.std():.4f}"
    )

ngram_df = pd.DataFrame(ngram_results)
ngram_df

Unigrams: 0.8942 +/- 0.0008
Unigrams + Bigrams: 0.9027 +/- 0.0009


,Representation,Mean CV Accuracy,Standard Deviation
0,Unigrams,0.894241,0.000759
1,Unigrams + Bigrams,0.902740,0.000874
